## Libraries and Packages

In [1]:
!pip install transformers datasets peft accelerate evaluate sacrebleu rouge_score sentencepiece -q

In [2]:
import json
import re
import unicodedata
import torch
import evaluate
import random
import gc
import os

import pandas as pd

from pandas import DataFrame
from typing import Dict, List, Optional, Any
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel
)
from dataclasses import dataclass
from sklearn.model_selection import train_test_split
from pathlib import Path
from datetime import datetime
from huggingface_hub import snapshot_download
from copy import deepcopy

W0802 21:09:07.187000 311184 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0802 21:09:07.210000 311184 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:
MODEL_NAME = "facebook/nllb-200-1.3B"
MODEL_DIRECTORY = Path("./models/facebook_nllb_1_3b")
EPOCHS = 6

In [4]:
def download_base_model() -> Path:

    if (
        MODEL_DIRECTORY.exists()
        and
        (MODEL_DIRECTORY / "config.json").exists()
    ):
        return MODEL_DIRECTORY


    MODEL_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True
    )


    snapshot_download(

        repo_id=MODEL_NAME,

        local_dir=str(
            MODEL_DIRECTORY
        ),

        local_dir_use_symlinks=False

    )

    return MODEL_DIRECTORY

In [5]:
class BaseModelCache:

    _tokenizer = None
    _model = None


    @classmethod
    def load(cls):

        model_path = download_base_model()

        if cls._tokenizer is None:

            cls._tokenizer = AutoTokenizer.from_pretrained(
                model_path,
                local_files_only=True
            )


        if cls._model is None:

            cls._model = AutoModelForSeq2SeqLM.from_pretrained(
                model_path,
                torch_dtype=torch.float16,
                device_map="auto",
                local_files_only=True
            )


    @classmethod
    def tokenizer(cls):

        cls.load()

        return cls._tokenizer


    @classmethod
    def model(cls):

        cls.load()

        return cls._model

## Classes and Functions

In [6]:
class PreprocessText:
    """
    Generic text preprocessing for multilingual datasets.
    """

    def __init__(self, text_columns: List[str]):
        self.text_columns = text_columns

    def _drop_empty_rows(self, df: DataFrame) -> DataFrame:
        """
        Remove rows where any specified text column is missing or empty.
        """
        clean_df = df.copy()

        for column in self.text_columns:
            clean_df[column] = clean_df[column].fillna("").astype(str)

        mask = clean_df[self.text_columns].apply(
            lambda row: all(cell.strip() != "" for cell in row),
            axis=1,
        )

        return clean_df.loc[mask].reset_index(drop=True)

    def _drop_duplicates(self, df: DataFrame) -> DataFrame:
        """
        Remove duplicate multilingual sentence pairs.
        """
        return (
            df.drop_duplicates(subset=self.text_columns)
              .reset_index(drop=True)
        )

    def _strip_whitespace(self, df: DataFrame) -> DataFrame:
        clean_df = df.copy()

        for column in self.text_columns:
            clean_df[column] = (
                clean_df[column]
                .astype(str)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
            )

        return clean_df

    def _normalize_text(self, df: DataFrame) -> DataFrame:
        clean_df = df.copy()

        def normalize(text: str) -> str:
            text = unicodedata.normalize("NFKC", text)
            text = re.sub(r"\s+", " ", text)
            text = re.sub(r"\s+([.,!?;:])", r"\1", text)
            return text.strip()

        for column in self.text_columns:
            clean_df[column] = clean_df[column].apply(normalize)

        return clean_df

    def clean_text(self, df: DataFrame) -> DataFrame:
        df = self._drop_empty_rows(df)
        df = self._strip_whitespace(df)
        df = self._normalize_text(df)
        df = self._drop_duplicates(df)

        return df

In [7]:
@dataclass
class TranslationDataset:
    """
    Container for a single translation direction.
    """

    source_language: str
    target_language: str

    train: pd.DataFrame
    validation: pd.DataFrame
    test: pd.DataFrame

In [8]:
class DatasetSplitter:
    """
    Creates train/validation/test datasets for multilingual
    bidirectional translation.

    Expected input dataframe:

    | English | Swahili | Gikuyu |
    |---------|---------|--------|
    | text    | text    | text   |

    """

    def __init__(
        self,
        train_size: float = 0.8,
        validation_size: float = 0.1,
        test_size: float = 0.1,
        random_state: int = 42,
    ):

        if train_size + validation_size + test_size != 1:
            raise ValueError(
                "Train, validation and test sizes must sum to 1"
            )

        self.train_size = train_size
        self.validation_size = validation_size
        self.test_size = test_size
        self.random_state = random_state


    def _split_dataframe(
        self,
        df: pd.DataFrame
    ):
        """
        Creates global train/validation/test splits.

        This split is shared by all language pairs.
        """

        train_df, temp_df = train_test_split(
            df,
            train_size=self.train_size,
            random_state=self.random_state,
            shuffle=True,
        )

        relative_test_size = (
            self.test_size /
            (self.validation_size + self.test_size)
        )

        validation_df, test_df = train_test_split(
            temp_df,
            test_size=relative_test_size,
            random_state=self.random_state,
            shuffle=True,
        )

        return (
            train_df.reset_index(drop=True),
            validation_df.reset_index(drop=True),
            test_df.reset_index(drop=True),
        )


    def _create_translation_pair(
        self,
        train_df: pd.DataFrame,
        validation_df: pd.DataFrame,
        test_df: pd.DataFrame,
        source_column: str,
        target_column: str,
        source_language: str,
        target_language: str,
    ) -> TranslationDataset:
        """
        Converts multilingual dataframe into a single
        source-target translation dataset.
        """

        def extract(df):

            return pd.DataFrame(
                {
                    "source_text": df[source_column],
                    "target_text": df[target_column],
                }
            )

        return TranslationDataset(
            source_language=source_language,
            target_language=target_language,

            train=extract(train_df),
            validation=extract(validation_df),
            test=extract(test_df),
        )


    def prepare(
        self,
        df: pd.DataFrame
    ) -> Dict[str, TranslationDataset]:
        """
        Creates all six translation directions.

        Returns:
            {
                "eng_to_swh": TranslationDataset,
                ...
            }
        """

        train_df, validation_df, test_df = (
            self._split_dataframe(df)
        )


        language_pairs = {

            "eng_to_swh": {
                "source_column": "English",
                "target_column": "Swahili",
                "source_language": "eng_Latn",
                "target_language": "swh_Latn",
            },

            "swh_to_eng": {
                "source_column": "Swahili",
                "target_column": "English",
                "source_language": "swh_Latn",
                "target_language": "eng_Latn",
            },

            "eng_to_kik": {
                "source_column": "English",
                "target_column": "Gikuyu",
                "source_language": "eng_Latn",
                "target_language": "kik_Latn",
            },

            "kik_to_eng": {
                "source_column": "Gikuyu",
                "target_column": "English",
                "source_language": "kik_Latn",
                "target_language": "eng_Latn",
            },

            "swh_to_kik": {
                "source_column": "Swahili",
                "target_column": "Gikuyu",
                "source_language": "swh_Latn",
                "target_language": "kik_Latn",
            },

            "kik_to_swh": {
                "source_column": "Gikuyu",
                "target_column": "Swahili",
                "source_language": "kik_Latn",
                "target_language": "swh_Latn",
            },
        }


        datasets = {}

        for name, config in language_pairs.items():

            datasets[name] = self._create_translation_pair(
                train_df=train_df,
                validation_df=validation_df,
                test_df=test_df,

                **config
            )


        return datasets

In [9]:
class NLLBLoRAFineTuner:
    """
    Fine-tunes NLLB-200 using LoRA.
    Handles one translation direction.
    """
    def __init__(
        self,
        model_name: str,
        source_lang: str,
        target_lang: str,
        train_data: pd.DataFrame,
        validation_data: pd.DataFrame,
        test_data: pd.DataFrame,
        output_dir: str,
        training_config: Optional[Dict] = None
    ):
        self.model_name = model_name
        self.source_lang = source_lang
        self.target_lang = target_lang

        self.train_data = train_data
        self.validation_data = validation_data
        self.test_data = test_data

        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.training_config = training_config or {}

        self.tokenizer = None
        self.model = None
        self.trainer = None

        self.bleu = evaluate.load("sacrebleu")
        self.rouge = evaluate.load("rouge")

    def load_tokenizer_and_model(self):
        # Load Tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.tokenizer.src_lang = self.source_lang
        self.tokenizer.tgt_lang = self.target_lang

        # Load Model
        base_model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
        
        # Attach LoRA
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=16,
            lora_alpha=32,
            lora_dropout=0.05,
            target_modules=["q_proj", "k_proj", "v_proj", "out_proj"]
        )
        self.model = get_peft_model(base_model, lora_config)
        self.model.print_trainable_parameters()

    def prepare_dataset(self, dataframe: pd.DataFrame) -> Dataset:
        return Dataset.from_pandas(dataframe)

    def tokenize_dataset(self, dataset: Dataset) -> Dataset:
        def tokenize(batch):
            self.tokenizer.src_lang = self.source_lang
            self.tokenizer.tgt_lang = self.target_lang

            inputs = self.tokenizer(
                batch["source_text"],
                max_length=128,
                truncation=True
            )
            labels = self.tokenizer(
                text_target=batch["target_text"],
                max_length=128,
                truncation=True
            )
            
            inputs["labels"] = labels["input_ids"]
            return inputs

        return dataset.map(
            tokenize,
            batched=True,
            remove_columns=dataset.column_names
        )

    def prepare_datasets(self):
        train = self.prepare_dataset(self.train_data)
        val = self.prepare_dataset(self.validation_data)
        
        return {
            "train": self.tokenize_dataset(train),
            "validation": self.tokenize_dataset(val)
        }

    def build_training_arguments(self):
        return Seq2SeqTrainingArguments(
            output_dir=str(self.output_dir),
            num_train_epochs=self.training_config.get("num_train_epochs", 6),
            learning_rate=2e-4,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            gradient_accumulation_steps=8,
            fp16=torch.cuda.is_available(),
            eval_strategy="epoch",  # Updated from evaluation_strategy
            save_strategy="epoch",
            logging_steps=50,
            report_to=["none"], # No MLflow, WandB, etc.
            predict_with_generate=False # We handle eval manually on the test set post-training
        )

    def build_trainer(self, datasets):
        return Seq2SeqTrainer(
            model=self.model,
            args=self.build_training_arguments(),
            train_dataset=datasets["train"],
            eval_dataset=datasets["validation"],
            processing_class=self.tokenizer,  # <--- CHANGED FROM tokenizer=self.tokenizer
            data_collator=DataCollatorForSeq2Seq(
                self.tokenizer, # DataCollator still expects the tokenizer here
                model=self.model
            )
        )

    def train(self):
        self.load_tokenizer_and_model()
        datasets = self.prepare_datasets()
        
        self.trainer = self.build_trainer(datasets)
        result = self.trainer.train()

        self.save_final_adapter()
        
        # Extract basic metrics
        metrics = {
            "train_loss": result.metrics.get("train_loss", result.training_loss),
            "epochs": self.training_config.get("num_train_epochs", 6)
        }

        # Cleanup memory for the next loop
        del self.trainer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return metrics

    def save_final_adapter(self):
        """Saves only the LoRA adapters and the tokenizer."""
        self.model.save_pretrained(self.output_dir)
        self.tokenizer.save_pretrained(self.output_dir)

    def translate(self, sentences: List[str]) -> List[str]:
        self.model.eval()
        device = self.model.device

        # Get Forced BOS Token ID dynamically
        forced_bos_token_id = self.tokenizer.convert_tokens_to_ids(self.target_lang)

        # Batch inputs
        inputs = self.tokenizer(
            sentences,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            generated = self.model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_new_tokens=128,
                max_length=None
            )

        return self.tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )

    def evaluate(self, test_df: pd.DataFrame) -> Dict:
        """Evaluates against the held-out test dataframe."""
        predictions = self.translate(test_df["source_text"].tolist())
        references = test_df["target_text"].tolist()

        # sacrebleu requires references to be list of lists: [[ref1], [ref2]]
        refs_for_bleu = [[ref] for ref in references]

        bleu_score = self.bleu.compute(
            predictions=predictions,
            references=refs_for_bleu
        )

        rouge_score = self.rouge.compute(
            predictions=predictions,
            references=references
        )

        return {
            "bleu": bleu_score["score"],
            "rouge1": rouge_score["rouge1"],
            "rougeL": rouge_score["rougeL"]
        }

    def run(self) -> Dict:
        metrics = self.train()
        return metrics


class NLLBTrainingPipeline:
    """
    Orchestrates training of multiple NLLB LoRA adapters sequentially.
    """
    def __init__(
        self,
        translation_datasets: Dict[str, TranslationDataset],
        model_name: str,
        epochs: int,
        output_root: str = "./nllb_lora_adapters"
    ):
        self.translation_datasets = translation_datasets
        self.model_name = model_name
        self.epochs = epochs
        self.output_root = Path(output_root)
        self.output_root.mkdir(parents=True, exist_ok=True)
        
        self.results = {}

    def _save_json(self, data: Dict, path: Path):
        with open(path, "w", encoding="utf-8") as file:
            json.dump(
                data,
                file,
                indent=4,
                ensure_ascii=False,
                default=str
            )

    def _create_metadata(self, adapter_name: str, dataset: TranslationDataset) -> Dict:
        return {
            "experiment": adapter_name,
            "base_model": self.model_name,
            "source_language": dataset.source_language,
            "target_language": dataset.target_language,
            "training_method": "LoRA",
            "epochs": self.epochs,
            "dataset": {
                "train_samples": len(dataset.train),
                "validation_samples": len(dataset.validation),
                "test_samples": len(dataset.test)
            },
            "created_at": datetime.utcnow().isoformat(),
            "status": "started"
        }

    def _build_training_config(self, output_dir: Path) -> Dict:
        return {
            "num_train_epochs": self.epochs,
            "output_dir": str(output_dir)
        }

    def _evaluate_adapter(self, tuner: NLLBLoRAFineTuner, dataset: TranslationDataset) -> Dict:
        """Runs evaluation on the test set."""
        print(f"Evaluating {dataset.source_language} -> {dataset.target_language}...")
        return tuner.evaluate(dataset.test)

    def _train_adapter(self, adapter_name: str, dataset: TranslationDataset) -> Dict:
        adapter_dir = self.output_root / adapter_name
        adapter_dir.mkdir(parents=True, exist_ok=True)

        metadata = self._create_metadata(adapter_name, dataset)
        self._save_json(metadata, adapter_dir / "training_metadata.json")

        tuner = NLLBLoRAFineTuner(
            model_name=self.model_name,
            source_lang=dataset.source_language,
            target_lang=dataset.target_language,
            train_data=dataset.train,
            validation_data=dataset.validation,
            test_data=dataset.test,
            output_dir=str(adapter_dir),
            training_config=self._build_training_config(adapter_dir)
        )

        # 1. Train
        training_metrics = tuner.run()

        # 2. Evaluate
        evaluation_metrics = self._evaluate_adapter(tuner, dataset)

        # Clean up heavy model loaded in memory after evaluation is complete
        del tuner.model
        del tuner.tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        metrics = {
            "training": training_metrics,
            "evaluation": evaluation_metrics
        }
        self._save_json(metrics, adapter_dir / "metrics.json")

        metadata.update({
            "status": "completed",
            "metrics_file": "metrics.json"
        })
        self._save_json(metadata, adapter_dir / "training_metadata.json")

        return {
            "adapter": adapter_name,
            "source": dataset.source_language,
            "target": dataset.target_language,
            "adapter_path": str(adapter_dir),
            "training": training_metrics,
            "evaluation": evaluation_metrics
        }

    def run(self) -> Dict[str, Any]:
        """Train all language adapters. Returns visualization-ready results."""
        for adapter_name, dataset in self.translation_datasets.items():
            print(f"\n================ Training {adapter_name} ================")
            try:
                result = self._train_adapter(adapter_name, dataset)
                self.results[adapter_name] = {
                    "status": "success",
                    **result
                }
            except Exception as error:
                print(f"Failed to train {adapter_name}: {error}")
                self.results[adapter_name] = {
                    "status": "failed",
                    "error": str(error)
                }

        return self.results

## Data Loading & Preprocessing

In [10]:
raw_df = pd.read_csv('validated_psas.csv')

In [11]:
text_preprocessor = PreprocessText(
    text_columns=["English", "Swahili", "Gikuyu"]
)

In [12]:
clean_df = text_preprocessor.clean_text(raw_df)

In [13]:
splitter = DatasetSplitter(
    train_size=0.8,
    validation_size=0.1,
    test_size=0.1,
)

In [14]:
translation_datasets = splitter.prepare(clean_df)

In [15]:
eng_swh = translation_datasets["eng_to_swh"]

print(eng_swh.source_language)
print(eng_swh.target_language)

eng_swh.train.head()

eng_Latn
swh_Latn


,source_text,target_text
0,Spark your career! Join electrical vocational ...,Fanya kazi yako iwe rahisi!
1,Grants are available for setting up a greenhou...,Hizo ni ziada za kuanzisha nyumba ya joto.
2,EACC appeals to farmers to support for communi...,EACC inawaomba wakulima kusaidia maendeleo ya ...
3,Governor Fatuma Achani urges residents of Kwal...,Gavana Fatuma Achani anawahimiza wakazi wa Kwa...
4,Keep shrubbery near entry points low to mainta...,Weka vichaka vya miti chini karibu na sehemu z...


## Model Training

In [16]:
random.seed(456)

In [17]:
pipeline = NLLBTrainingPipeline(
    translation_datasets=translation_datasets,
    model_name=MODEL_NAME,
    epochs=EPOCHS,
    output_root="./nllb_lora_adapters"
)

In [18]:
results = pipeline.run()


================ Training eng_to_swh ================


/tmp/ipykernel_311184/1826439822.py:251: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

trainable params: 9,437,184 || all params: 1,380,075,520 || trainable%: 0.6838


Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

...

trainable params: 9,437,184 || all params: 1,380,075,520 || trainable%: 0.6838


Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Map:   0%|          | 0/184 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss
1,17.272085,1.692541
2,14.115527,1.646438
3,13.127737,1.618374
4,12.521095,1.607455
5,12.109642,1.605209
6,12.176227,1.601428


Evaluating kik_Latn -> swh_Latn...


In [19]:
metrics_df = pd.DataFrame(
    [
        {
            "adapter": k,
            **v["evaluation"]
        }
        for k,v in results.items()
        if v["status"] == "success"
    ]
)

In [20]:
metrics_df.head()

,adapter,bleu,rouge1,rougeL
0,eng_to_swh,53.917727,0.729661,0.711829
1,swh_to_eng,44.901412,0.712016,0.695034
2,eng_to_kik,17.862814,0.500714,0.462217
3,kik_to_eng,20.065999,0.451336,0.435765
4,swh_to_kik,14.814608,0.460891,0.419108
